In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica guiada por FEATURES + ajuste físico limitado
(com preservação explícita de dano)

+ (ADICIONADO) Gráfico RMSD e CCDM por temperatura e dano (Original vs Features),
no estilo do exemplo.
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

# ================================================================
# ========================= PARÂMETROS ============================
# ================================================================
ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30
TEMP_DESEJADA = 78
FALHA_DESEJADA = None

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

OVERLAP_MIN = 0.60
NSHIFTS = 201
TILT_GRID = 41
TILT_MAX = 0.4
SMOOTH_WIN_OUT = 5

W_TEMP = 1.0
W_PRES = 1.0

N_TEMP_FEATURES = 6
N_DMG_FEATURES = 6

# --------- parâmetros da varredura "em lote" (para gráfico global) ----------
# (deixe menor pra rodar rápido; se quiser mais preciso, aumente)
BATCH_NSHIFTS = 101
BATCH_TILT_GRID = 31

EPS = 1e-12

# ================================================================
# ===================== AUX FUNÇÕES ==============================
# ================================================================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def shift_by_samples(x, k):
    x = np.asarray(x)
    n = len(x)
    y = np.empty_like(x)
    if k == 0:
        return x.copy()
    if k > 0:
        y[:k] = x[0]
        y[k:] = x[:n-k]
    else:
        kk = -k
        y[n-kk:] = x[-1]
        y[:n-kk] = x[kk:]
    return y

def moving_average(arr, win):
    arr = np.asarray(arr, float)
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win)/win
    return np.convolve(arr_pad, kernel, mode="valid")[:len(arr)]

# ================================================================
# ===================== FEATURES AVANÇADAS =======================
# ================================================================
def extract_features(y, fhz):
    y = np.asarray(y, float)
    fk = fhz / 1e3

    mu = np.mean(y)
    sd = np.std(y)
    ptp = np.ptp(y)
    rms = np.sqrt(np.mean(y**2))

    x = fk - np.mean(fk)
    slope = np.dot(x, y-mu) / (np.dot(x,x)+EPS)

    energy = np.sum((y-mu)**2)
    centroid = np.sum(fk*np.abs(y-mu))/(np.sum(np.abs(y-mu))+EPS)

    skew = np.mean(((y-mu)/(sd+EPS))**3)
    kurt = np.mean(((y-mu)/(sd+EPS))**4)

    dy = np.diff(y)
    grad_energy = np.mean(dy**2) if len(dy) else 0.0

    d2 = np.diff(y,2)
    roughness = np.mean(d2**2) if len(d2) else 0.0

    idx = np.where((y[1:-1]>y[:-2])&(y[1:-1]>y[2:]))[0]+1
    peak_count = len(idx)
    peak_mean_amp = np.mean(y[idx]) if peak_count>0 else mu
    peak_max_amp = np.max(y[idx]) if peak_count>0 else mu
    peak_pos_mean = np.mean(fk[idx]) if peak_count>0 else np.mean(fk)

    bw = np.sqrt(np.sum(((fk-centroid)**2)*np.abs(y-mu))/(np.sum(np.abs(y-mu))+EPS))

    return {
        "mean":mu,
        "std":sd,
        "ptp":ptp,
        "rms":rms,
        "slope":slope,
        "energy":energy,
        "centroid":centroid,
        "skew":skew,
        "kurtosis":kurt,
        "grad_energy":grad_energy,
        "roughness":roughness,
        "peak_count":peak_count,
        "peak_mean_amp":peak_mean_amp,
        "peak_max_amp":peak_max_amp,
        "peak_pos_mean":peak_pos_mean,
        "bandwidth":bw
    }

# ================================================================
# ===================== RANK TEMP vs DANO ========================
# ================================================================
def effect_damage(df_feat, f):
    x = df_feat[f].values
    g = df_feat["falha"].values
    mu = np.mean(x)
    ss_tot = np.sum((x-mu)**2)+EPS
    ss_between = sum(len(x[g==c])*(np.mean(x[g==c])-mu)**2 for c in np.unique(g))
    return ss_between/ss_tot

def effect_temp(df_feat, f):
    vals=[]
    for d in np.unique(df_feat["falha"]):
        sub=df_feat[df_feat["falha"]==d]
        if len(sub)<3: 
            continue
        x=sub[f].values
        t=sub["temperatura_c"].values
        if np.std(x)<1e-9: 
            continue
        vals.append(abs(np.corrcoef(x,t)[0,1]))
    return np.mean(vals) if vals else 0

def rank_features(df_feat, features):
    rows=[]
    for f in features:
        sd=effect_damage(df_feat,f)
        st=effect_temp(df_feat,f)
        rows.append({
            "feature":f,
            "score_temp":st,
            "score_damage":sd,
            "ratio_temp_over_damage":st/(sd+1e-6),
            "ratio_damage_over_temp":sd/(st+1e-6)
        })
    return pd.DataFrame(rows)

# ================================================================
# =================== MÉTRICAS RMSD / CCDM =======================
# ================================================================
def calc_rmsd(ref, cur):
    ref = np.asarray(ref, float)
    cur = np.asarray(cur, float)
    return float(np.sqrt(np.mean((cur - ref)**2)))

def calc_ccdm(ref, cur):
    ref = np.asarray(ref, float)
    cur = np.asarray(cur, float)
    ref = (ref - np.mean(ref)) / (np.std(ref) + EPS)
    cur = (cur - np.mean(cur)) / (np.std(cur) + EPS)
    corr = np.correlate(ref, cur, mode='full')
    max_corr = np.max(np.abs(corr))
    return float(1 - max_corr/len(ref))

# ================================================================
# ============================ SCRIPT =============================
# ================================================================
df=pd.read_pickle(ARQ_BASE)
fcols,fhz=get_freq_columns(df,FREQ_MIN_KHZ,FREQ_MAX_KHZ)

# referência (dano 0 em REF_TEMP; fallback para mediana do dano 0)
df_sem=df[df["falha"]==0]
pool=df_sem[np.isclose(df_sem["temperatura_c"], REF_TEMP)][fcols].values
if len(pool) == 0:
    y_ref = np.median(df_sem[fcols].values, axis=0)
else:
    y_ref=np.median(pool,axis=0)

# curva escolhida (para o plot que você já tinha)
df_sel=df[np.isclose(df["temperatura_c"], TEMP_DESEJADA)]
if FALHA_DESEJADA is not None:
    df_sel=df_sel[df_sel["falha"]==FALHA_DESEJADA]
idx=df_sel.index[0]
y_orig=df.loc[idx,fcols].values
temp_real=float(df.loc[idx,"temperatura_c"])
falha_real=int(df.loc[idx,"falha"])

# extrai features da base toda
rows=[]
for _,row in df.iterrows():
    feats=extract_features(row[fcols].values,fhz)
    feats["temperatura_c"]=float(row["temperatura_c"])
    feats["falha"]=int(row["falha"])
    rows.append(feats)

df_feat=pd.DataFrame(rows)
feature_list=[c for c in df_feat.columns if c not in ["temperatura_c","falha"]]

rank=rank_features(df_feat,feature_list)

feat_temp=rank.sort_values("ratio_temp_over_damage",ascending=False)["feature"].head(N_TEMP_FEATURES).tolist()
feat_dmg=rank.sort_values("ratio_damage_over_temp",ascending=False)["feature"].head(N_DMG_FEATURES).tolist()

print("\nFeatures térmicas:",feat_temp)
print("Features de dano:",feat_dmg)

# ================================================================
# =================== AJUSTE FÍSICO SIMPLES ======================
# ================================================================
fk_centered = (fhz/1e3 - np.mean(fhz/1e3))

def apply_transform(y,shift,offset,gain,tilt):
    y2=shift_by_samples(y,shift)
    return gain*(y2+offset)+tilt*fk_centered

target_ref_feats=extract_features(y_ref,fhz)

def compensate_one_curve(y_in, nshifts=NSHIFTS, tilt_grid=TILT_GRID):
    """
    Compensa UMA curva pelo método de features (preserva as features de dano do próprio sinal).
    """
    y_in = np.asarray(y_in, float)
    orig_feats = extract_features(y_in, fhz)

    bestJ = 1e99
    best_y = None

    k_max=int((1-OVERLAP_MIN)*(len(y_in)-1))
    ks=np.linspace(-k_max,k_max,nshifts).round().astype(int)
    tilts=np.linspace(-TILT_MAX,TILT_MAX,tilt_grid)

    for k in ks:
        for tilt in tilts:

            ytmp=apply_transform(y_in,k,0,1,tilt)

            mu=np.mean(ytmp)
            sd=np.std(ytmp)+EPS
            gain=target_ref_feats["std"]/sd
            offset=(target_ref_feats["mean"]-gain*mu)/gain

            y_c=apply_transform(y_in,k,offset,gain,tilt)

            feats=extract_features(y_c,fhz)

            e_temp=sum((feats[f]-target_ref_feats[f])**2 for f in feat_temp)
            e_pres=sum((feats[f]-orig_feats[f])**2 for f in feat_dmg)

            J=W_TEMP*e_temp+W_PRES*e_pres

            if J<bestJ:
                bestJ=J
                best_y=y_c

    return moving_average(best_y, SMOOTH_WIN_OUT)

# ---------- compensa a curva escolhida (plot antigo) ----------
y_comp = compensate_one_curve(y_orig, nshifts=NSHIFTS, tilt_grid=TILT_GRID)

# ================================================================
# ============================ PLOT 1 (o seu) =====================
# ================================================================
plt.rcParams.update({"font.size": 14, "font.family": "Times New Roman", "text.usetex": False})

plt.figure(figsize=(12,6),dpi=300)
plt.plot(fhz/1e3,y_ref,"--",color="black",label=f"Ref {REF_TEMP}°C")
plt.plot(fhz/1e3,y_orig,color="red",alpha=0.6,label=f"Original {temp_real:.0f}°C")
plt.plot(fhz/1e3,y_comp,color="blue",lw=2,label="Compensado (features)")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real")
plt.title(f"Compensação por Features (Temp={temp_real:.0f}°C, Dano={falha_real})")
plt.legend(frameon=True, edgecolor="none")
plt.grid(False)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# ================================================================
# ===== PLOT 2 (novo): RMSD e CCDM por temperatura e dano =========
# ================================================================
# Temperaturas "válidas": onde existem os 3 danos (0,1,2)
temps = np.sort(df["temperatura_c"].unique())
valid_temps = []
for T in temps:
    sub = df[np.isclose(df["temperatura_c"], T)]
    if set(sub["falha"].unique()) >= {0,1,2}:
        valid_temps.append(float(T))
valid_temps = np.array(valid_temps, float)

if len(valid_temps) == 0:
    raise RuntimeError("Não encontrei temperaturas com os 3 danos (0,1,2) ao mesmo tempo.")

# Calcula métricas médias por temperatura e dano
# Para o gráfico global, uso a busca 'em lote' (mais leve) por curva
metrics = {
    "Original": {"RMSD": {0: [], 1: [], 2: []}, "CCDM": {0: [], 1: [], 2: []}},
    "Features": {"RMSD": {0: [], 1: [], 2: []}, "CCDM": {0: [], 1: [], 2: []}},
}

for T in valid_temps:
    for d in [0,1,2]:
        sub = df[np.isclose(df["temperatura_c"], T) & (df["falha"]==d)]
        X = sub[fcols].values.astype(float)

        # --- Original: direto ---
        rmsd_vals = [calc_rmsd(y_ref, x) for x in X]
        ccdm_vals = [calc_ccdm(y_ref, x) for x in X]
        metrics["Original"]["RMSD"][d].append(float(np.mean(rmsd_vals)))
        metrics["Original"]["CCDM"][d].append(float(np.mean(ccdm_vals)))

        # --- Features: compensa cada curva e mede ---
        comp_vals = []
        for x in X:
            yx = compensate_one_curve(x, nshifts=BATCH_NSHIFTS, tilt_grid=BATCH_TILT_GRID)
            comp_vals.append(yx)

        rmsd_c = [calc_rmsd(y_ref, yx) for yx in comp_vals]
        ccdm_c = [calc_ccdm(y_ref, yx) for yx in comp_vals]
        metrics["Features"]["RMSD"][d].append(float(np.mean(rmsd_c)))
        metrics["Features"]["CCDM"][d].append(float(np.mean(ccdm_c)))

# ---------- Plot estilo exemplo: 2 subplots (RMSD e CCDM) ----------
x = np.arange(len(valid_temps))
bar_w = 0.10

# 6 barras por temperatura: Original(d0,d1,d2) + Features(d0,d1,d2)
offsets = {
    ("Original",0): -2.5*bar_w,
    ("Original",1): -1.5*bar_w,
    ("Original",2): -0.5*bar_w,
    ("Features",0): +0.5*bar_w,
    ("Features",1): +1.5*bar_w,
    ("Features",2): +2.5*bar_w,
}

# cores (padrão matplotlib)
colors = {
    ("Original",0): "tab:blue",
    ("Original",1): "tab:orange",
    ("Original",2): "tab:red",
    ("Features",0): "tab:blue",
    ("Features",1): "tab:orange",
    ("Features",2): "tab:red",
}
alphas = {"Original": 0.35, "Features": 0.85}

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300, sharex=True)
fig.suptitle("Original vs Features — Temperaturas válidas (3 danos em ambos)", y=1.02)

# --- RMSD ---
ax = axes[0]
for method in ["Original","Features"]:
    for d in [0,1,2]:
        ax.bar(
            x + offsets[(method,d)],
            metrics[method]["RMSD"][d],
            width=bar_w,
            color=colors[(method,d)],
            alpha=alphas[method],
            label=f"{method} — Dano {d}"
        )
ax.set_title(f"RMSD (REF={REF_TEMP} °C)")
ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("RMSD")
ax.set_xticks(x)
ax.set_xticklabels([f"{t:.1f}" for t in valid_temps])
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- CCDM ---
ax = axes[1]
for method in ["Original","Features"]:
    for d in [0,1,2]:
        ax.bar(
            x + offsets[(method,d)],
            metrics[method]["CCDM"][d],
            width=bar_w,
            color=colors[(method,d)],
            alpha=alphas[method],
            label=f"{method} — Dano {d}"
        )
ax.set_title(f"CCDM (REF={REF_TEMP} °C)")
ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("CCDM")
ax.set_xticks(x)
ax.set_xticklabels([f"{t:.1f}" for t in valid_temps])
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# legenda única em cima
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)

plt.tight_layout()
plt.show()